# MOSS-Image on Google Cloud TPU (`torch_tpu` / TorchTPU)

This notebook runs **OpenMOSS/MOSS-Image-Base** (Qwen3.5-9B text encoder + 5B single-stream DiT + FLUX.2 VAE) on **Google Colab `TPU v6e-1` (32 GiB HBM)** using Google's native PyTorch backend (`torch_tpu`).

**Before running:** Select `Runtime -> Change runtime type -> v6e-1 TPU` (requires Python 3.11+).

In [ ]:
import os

os.makedirs("/content/tpu_cache/tier2", exist_ok=True)
os.makedirs("/content/tpu_cache/tier3", exist_ok=True)
os.environ["TORCH_TPU_TIER2_COMPILATION_CACHE"] = "/content/tpu_cache/tier2"
os.environ["TORCH_TPU_TIER3_COMPILATION_CACHE_ROOT"] = "/content/tpu_cache/tier3"

!git clone https://github.com/azure-swing/MOSS-Image.git || true
%cd /content/MOSS-Image
!git checkout feat/torchtpu-support || true
!bash scripts/setup-tpu.sh

## 1. Probe TPU & `torch_tpu` Runtime
Verify `device="tpu"`, HBM capacity, and BF16 MXU matmul execution.

In [ ]:
!python3 scripts/run_tpu.py --mode probe

## 2. Generate Image (Staged Qwen3.5-9B + Static-Bucket Compiled 5B DiT)
- **Stage 1 (`text_encoder`)** runs unpadded individual prompt forwards (preserving Qwen3.5 GDN numerics) and caches embeddings on CPU.
- **Stage 2 (`transformer` + `vae`)** pads embeddings to fixed static buckets (`512/1024/2048/4096`) with SDPA attention masking and compiles with `torch.compile(..., backend="tpu", dynamic=False, fullgraph=True)`.

In [ ]:
import time
import torch
from IPython.display import display
import tpu_runtime

pipe = tpu_runtime.load_pipeline(
    "OpenMOSS-Team/MOSS-Image-Base",
    offload="model",
    compile_transformer=True,
    prompt_buckets=(512, 1024, 2048, 4096),
    matmul_precision="high",
)

prompt = (
    "A sleek editorial poster on matte cream paper with crisp black serif typography reading "
    "'TORCHTPU X MOSS-IMAGE 2026' above a minimalist ceramic teapot, studio daylight."
)

for run_idx in (1, 2):
    tpu_runtime.reset_peak_memory_stats()
    t0 = time.perf_counter()
    image = pipe(
        prompt=prompt,
        height=1024,
        width=1024,
        num_inference_steps=28,
        guidance_scale=3.0,
        generator=torch.Generator("cpu").manual_seed(42 + run_idx),
    ).images[0]
    tpu_runtime.synchronize()
    elapsed = time.perf_counter() - t0
    peak = tpu_runtime.max_memory_allocated_gib()
    label = "cold (includes XLA compile)" if run_idx == 1 else "warm (cached static graph)"
    print(f"Run {run_idx} [{label}]: {elapsed:.2f}s | peak HBM: {peak:.2f} GiB")

display(image)
image.save("outputs/colab_tpu_sample.png")

## 3. Benchmark: Strict Eager vs `DEFER_AND_FUSE` vs `torch.compile(backend="tpu")`

In [ ]:
!python3 scripts/run_tpu.py \
  --mode benchmark \
  --bench-modes eager,defer_and_fuse,compile \
  --height 1024 \
  --width 1024 \
  --steps 28 \
  --runs 2 \
  --save-report outputs/tpu_benchmark.json

## 4. Optional: Decoupled Stage 1 / Stage 2 Workflow (for 16 GiB TPU v5e-1)

In [ ]:
# !python3 scripts/run_tpu.py --mode encode-only --prompt "A neon rainy Tokyo alley" --save-embeds outputs/embeds.pt
# !python3 scripts/run_tpu.py --mode generate --prompt-embeds outputs/embeds.pt --offload none --compile --output outputs/decoupled.png